# Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly
import plotly.express as px
import seaborn as sns
import joblib
from IPython.core.display import HTML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn import metrics
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from datetime import datetime
from sklearn.feature_extraction.text import CountVectorizer
from pandas import DataFrame
from collections import OrderedDict 
from colorama import Fore, Back, Style
y_ = Fore.YELLOW
r_ = Fore.RED
g_ = Fore.GREEN
b_ = Fore.BLUE
m_ = Fore.MAGENTA
sr_ = Style.RESET_ALL

# Reading the csv file

In [ ]:
df = pd.read_csv(r'news_articles.csv', encoding="latin", index_col=0)
df = df.dropna()
df.count()

In [ ]:
real = df["label"].loc[df["label"] == "Real"].count()
fake = df["label"].loc[df["label"] == "Fake"].count()

print("No. of real new :", real)
print("No. of fake new :", fake)

In [ ]:
df.head(5)

In [ ]:
df['type'].unique()

# Distrubution of types of articles

In [ ]:
df['type'].value_counts().plot.pie(figsize = (8,8), startangle = 75)
plt.title('Types of articles', fontsize = 20)
plt.axis('off')
plt.show()

# Unigrams and bigrams 

In [ ]:
def get_top_n_words(corpus, n=None):
    vec = CountVectorizer().fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0) 
    words_freq = [(word, sum_words[0, idx]) for word, idx in     vec.vocabulary_.items()]
    words_freq =sorted(words_freq, key = lambda x: x[1], reverse=True)
    return words_freq[:n]

def get_top_n_bigram(corpus, n=None):
    vec = CountVectorizer(ngram_range=(2, 2)).fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0) 
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq =sorted(words_freq, key = lambda x: x[1], reverse=True)
    return words_freq[:n]


def get_top_n_trigram(corpus, n=None):
    vec = CountVectorizer(ngram_range=(3, 3)).fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0) 
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq =sorted(words_freq, key = lambda x: x[1], reverse=True)
    return words_freq[:n]

In [ ]:
common_words = get_top_n_words(df['text_without_stopwords'], 20)
df3 = pd.DataFrame(common_words, columns=['words', 'count'])

df3.groupby('words').sum()['count'].sort_values(ascending=False).plot(
    kind='bar',
    ylabel='Count',  
    title='Top 20 bigrams used in articles',
    color='blue',
    edgecolor='black' 
)


In [ ]:
common_words = get_top_n_bigram(df['text_without_stopwords'], 20)
df3 = pd.DataFrame(common_words, columns=['words', 'count'])

df3.groupby('words').sum()['count'].sort_values(ascending=False).plot(
    kind='bar',
    ylabel='Count',  
    title='Top 20 bigrams used in articles',
    color='blue',
    edgecolor='black' 
)


# WordCloud of articles

# Articles including images vs Label

In [ ]:
def convert(path):
    return '<img src="'+ path + '" width="80">'

In [ ]:
df_sources = df[['site_url','label','main_img_url']]
df_r = df_sources.loc[df['label']== 'Real'].iloc[6:10,:]
df_f = df_sources.loc[df['label']== 'Fake'].head(6)

In [ ]:
HTML(df_r.to_html(escape=False,formatters=dict(main_img_url=convert)))

In [ ]:
HTML(df_f.to_html(escape=False,formatters=dict(main_img_url=convert)))

In [ ]:
df['site_url'].unique()

In [ ]:
type_label = {'Real': 0, 'Fake': 1}
df_sources.label = [type_label[item] for item in df_sources.label] 

In [ ]:
val_real=[]
val_fake=[]

for i,row in df_sources.iterrows():
    val = row['site_url']
    if row['label'] == 0:
        val_real.append(val)
    elif row['label']== 1:
        val_fake.append(val)

> # Websites publishing real news

In [ ]:
uniqueValues_real = list(OrderedDict.fromkeys(val_real)) 

print(f"{y_}Websites publishing real news:{g_}{uniqueValues_real}\n") 

# Websites publishing fake news

In [ ]:
uniqueValues_fake = list(OrderedDict.fromkeys(val_fake)) 
print(f"{y_}Websites publishing fake news:{r_}{uniqueValues_fake}\n")

# Websites publishing both real and fake news

In [ ]:
real_set = set(uniqueValues_real) 
fake_set = set(uniqueValues_fake) 

print(f"{y_}Websites publishing both real and fake news:{m_}{real_set & fake_set}\n")

In [ ]:
type1 = {'bias': 0, 'conspiracy': 1,'fake': 2,'bs': 3,'satire': 4, 'hate': 5,'junksci': 6, 'state': 7}
df.type = [type1[item] for item in df.type] 

In [ ]:
def plot_bar(df, feat_x, feat_y, normalize=True):
    """ Plot with vertical bars of the requested dataframe and features"""
    
    ct = pd.crosstab(df[feat_x], df[feat_y])
    if normalize == True:
        ct = ct.div(ct.sum(axis=1), axis=0)
    return ct.plot(kind='bar', stacked=True)

# Label vs Type

In [ ]:
plot_bar(df,'type' , 'label')
plt.show()

# Websites and types of news published

In [ ]:
df_type = df[['site_url','type']]

val_bias=[]
val_conspiracy=[]
val_fake1=[]
val_bs=[]
val_satire=[]
val_hate=[]
val_junksci=[]
val_state=[]
{'bias': 0, 'conspiracy': 1,'fake': 2,'bs': 3,'satire': 4, 'hate': 5,'junksci': 6, 'state': 7}
for i,row in df_type.iterrows():
    val = row['site_url']
    if row['type'] == 0:
        val_bias.append(val)
    elif row['type']== 1:
        val_conspiracy.append(val)
    elif row['type']== 2:
        val_fake1.append(val)
    elif row['type']== 3:
        val_bs.append(val)
    elif row['type']== 4:
        val_satire.append(val)
    elif row['type']== 5:
        val_hate.append(val)
    elif row['type']== 6:
        val_junksci.append(val)
    elif row['type']== 7:
        val_state.append(val)

In [ ]:
uv_bias = list(OrderedDict.fromkeys(val_bias)) 
uv_conspiracy = list(OrderedDict.fromkeys(val_conspiracy)) 
uv_fake = list(OrderedDict.fromkeys(val_fake1)) 
uv_bs = list(OrderedDict.fromkeys(val_bs)) 
uv_satire = list(OrderedDict.fromkeys(val_satire)) 
uv_hate = list(OrderedDict.fromkeys(val_hate)) 
uv_junksci = list(OrderedDict.fromkeys(val_junksci)) 
uv_state = list(OrderedDict.fromkeys(val_state)) 

print(f"{b_}{type1}\n")
i=0
for lst in (uv_bias,uv_conspiracy,uv_fake,uv_bs,uv_satire, uv_hate,uv_junksci,uv_state): 
    print(f"{y_}Source URLs for type:{b_}{i}{r_}{lst}\n") 
    i+=1

# Shuffling values

In [ ]:
df1 = df.sample(frac=1)
df1.head()

# Training and Testing

In [134]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn import metrics
from keras.models import Sequential
import joblib
from keras.layers import Dense, Dropout, BatchNormalization
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:

# Prepare features and labels
y = df['label']
x = df.loc[:, ['site_url', 'text_without_stopwords']]
x['source'] = x["site_url"].astype(str) + " " + x["text_without_stopwords"]
x = x.drop(['site_url', 'text_without_stopwords'], axis=1)
x = x.source

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

# Split the data into training and test sets (80% training, 20% test)
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)


In [ ]:

# Use TF-IDF to vectorize the text data
tfidf_vect = TfidfVectorizer(stop_words='english')
tfidf_train = tfidf_vect.fit_transform(X_train)
tfidf_test = tfidf_vect.transform(X_test)


In [135]:
# Build a deeper neural network model
model = Sequential()

# Input layer and first hidden layer (increased neurons and added dropout)
model.add(Dense(1024, input_dim=tfidf_train.shape[1], activation='relu'))  # Increased neurons
model.add(Dropout(0.5))  # Added dropout to prevent overfitting

# Added more hidden layers to capture complex patterns
model.add(Dense(512, activation='relu'))  # Second hidden layer with more neurons
model.add(Dropout(0.5))  # Dropout for regularization

model.add(Dense(256, activation='relu'))  # Third hidden layer
model.add(Dropout(0.5))

# Adding Batch Normalization to help with training stability
model.add(BatchNormalization())

# Output layer (binary classification with sigmoid activation)
model.add(Dense(1, activation='sigmoid'))

# Compile the model with Adam optimizer and learning rate scheduler
model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.0005), metrics=['accuracy'])  # Increased learning rate

# Use EarlyStopping and ModelCheckpoint to avoid overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)  # Increased patience
checkpoint = ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True)

# Train the model with validation split and callbacks
model.fit(tfidf_train, y_train, epochs=100, batch_size=64, verbose=1, 
          validation_data=(tfidf_test, y_test), callbacks=[early_stopping, checkpoint])

# Save the trained model
model.save('shallow_neural_network_model_keras_improved_v2.keras')

# Evaluate the model on the test set
y_pred_nn = (model.predict(tfidf_test) > 0.5).astype("int32")

# Print accuracy score
NNscore = accuracy_score(y_test, y_pred_nn)
print("Neural Network Accuracy:  %0.3f" % NNscore)

# Print Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_nn))

# Print Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_nn))

c:\Users\KIIT\Desktop\ranveer\Sem7Project\.venv\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



Epoch 1/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 17s 522ms/step - accuracy: 0.5355 - loss: 0.6858 - val_accuracy: 0.6455 - val_loss: 0.6584
Epoch 2/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 13s 493ms/step - accuracy: 0.8233 - loss: 0.4208 - val_accuracy: 0.6528 - val_loss: 0.6015
Epoch 3/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 13s 492ms/step - accuracy: 0.9757 - loss: 0.0941 - val_accuracy: 0.6773 - val_loss: 0.5684
Epoch 4/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 13s 490ms/step - accuracy: 0.9919 - loss: 0.0396 - val_accuracy: 0.7042 - val_loss: 0.5463
Epoch 5/100
26/26 ━━━━━━━━━━━━━━━━━━━━ 13s 501ms/step - accuracy: 0.9962 - loss: 0.0264 - val_accuracy: 0.7384 - val_loss: 0.5273
Epoch 6/100
15/26 ━━━━━━━━━━━━━━━━━━━━ 4s 397ms/step - accuracy: 0.9987 - loss: 0.0151

KeyboardInterrupt: 

In [ ]:

# Save the trained model
model.save('shallow_neural_network_model_keras.h5')


In [126]:
import joblib

# Save the fitted tfidf vectorizer after training
joblib.dump(tfidf_vect, 'tfidf_vectorizer.pkl')


['tfidf_vectorizer.pkl']

In [130]:
from keras.models import load_model
import joblib
import numpy as np

# Load the trained model
saved_model_nn = load_model('shallow_neural_network_model_keras.h5')

# Load the same TfidfVectorizer used during training
tfidf_vect = joblib.load('tfidf_vectorizer.pkl')  # Load the same vectorizer used during training

# New article for prediction
new_article = ["Donald Trump won the us elections"]

# Apply the same TF-IDF transformation to the new article
new_article_tfidf = tfidf_vect.transform(new_article)

# Check if the shape of the new article matches the expected shape (42005)
print(f"New article TF-IDF shape: {new_article_tfidf.shape}")

# The model was trained with 42005 features, so we need to ensure the new article has the same shape
expected_shape = (1, 42005)  # Model expects this shape

# If the shape doesn't match, manually pad or truncate the input to match the expected shape
if new_article_tfidf.shape[1] != expected_shape[1]:
    # Manually pad the feature vector with zeros if the length is shorter than expected
    padded_article_tfidf = np.pad(new_article_tfidf.toarray(), ((0, 0), (0, expected_shape[1] - new_article_tfidf.shape[1])), 'constant', constant_values=0)
    print(f"Shape after padding: {padded_article_tfidf.shape}")
else:
    padded_article_tfidf = new_article_tfidf.toarray()

# Making predictions with the model
probabilities_nn = saved_model_nn.predict(padded_article_tfidf)
print("Probabilities: ", probabilities_nn)

# Convert the probability to a binary class (e.g., if the probability is greater than 0.5, classify as 1)
predictions_nn = (probabilities_nn > 0.5).astype("int32")
print("Predicted Class (0 or 1): ", predictions_nn)


New article TF-IDF shape: (1, 38600)
Shape after padding: (1, 42005)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
Probabilities:  [[0.13200456]]
Predicted Class (0 or 1):  [[0]]


Right after preprocessing, the output is a corpus of raw texts that are stripped of stopwords, stemmed and lemmatized. 

In order to get a sparse matrix of TF/IDF values, the following steps are taken:
* Tokenization of texts
* Counting of the tokens and
* Transforming the raw tokens into TF/IDF values

The above steps are done with the help of the TfidfVectorizer, which transforms text to feature vectors that can be used
as input to estimators/classifiers.